In [7]:
import numpy as np
import pandas as pd
import os
import gc

!pip install mlflow dagshub -q

from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
os.environ["MLFLOW_TRACKING_PASSWORD"] = user_secrets.get_secret("DAGSHUB_TOKEN")
os.environ["MLFLOW_TRACKING_USERNAME"] = user_secrets.get_secret("DAGSHUB_USERNAME")

import mlflow
mlflow.set_tracking_uri("https://dagshub.com/llikl23/IEEE-CIS-Fraud-Detection.mlflow")
mlflow.set_experiment("LogisticRegression_Training")

print("MLflow connected!")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.2/49.2 kB 2.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.0/50.0 kB 3.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 98.7 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 97.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 68.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 273.1/273.1 kB 18.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.2/68.2 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 8.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 208.4/208.4 kB 16.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.0/77.0 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.2/132.2 kB 9.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [2]:
DATA_DIR = "/kaggle/input/competitions/ieee-fraud-detection"

train_transaction = pd.read_csv(f"{DATA_DIR}/train_transaction.csv")
train_identity = pd.read_csv(f"{DATA_DIR}/train_identity.csv")

train = train_transaction.merge(train_identity, on="TransactionID", how="left")

del train_transaction, train_identity
gc.collect()

print(f"Train shape: {train.shape}")
print(f"Fraud rate: {train['isFraud'].mean():.4f} ({train['isFraud'].sum()} / {len(train)})")
print(f"Columns: {train.shape[1]}")

Train shape: (590540, 434)
Fraud rate: 0.0350 (20663 / 590540)
Columns: 434


# EDA
პირველ რიგში, შევხედოთ მონაცემებს და გავარკვიოთ რამდენია კატეგორიული ცვლადი, რამდენია ცარიელი, რომელ ფიჩერებს აქვთ ყველაზე მეტი ცარიელი მნიშვნელობა

In [3]:
num_cols = train.select_dtypes(include=[np.number]).columns.tolist()
cat_cols = train.select_dtypes(include=["object"]).columns.tolist()
print(f"Numerical columns: {len(num_cols)}")
print(f"Categorical columns: {len(cat_cols)}")

missing = train.isnull().sum()
missing_pct = (missing / len(train) * 100).round(2)
missing_df = (
    pd.DataFrame({"n_missing": missing, "pct_missing": missing_pct})
    .query("n_missing > 0")
    .sort_values("pct_missing", ascending=False)
)
print(f"\nColumns with any missing values: {len(missing_df)} out of {train.shape[1]}")
print(f"\nColumns with >50% missing:")
print(missing_df[missing_df["pct_missing"] > 50].shape[0])
print(f"\nTop 20 most missing:")
print(missing_df.head(20))

Numerical columns: 403
Categorical columns: 31

Columns with any missing values: 414 out of 434

Columns with >50% missing:
214

Top 20 most missing:
       n_missing  pct_missing
id_24     585793        99.20
id_26     585377        99.13
id_25     585408        99.13
id_21     585381        99.13
id_07     585385        99.13
id_08     585385        99.13
id_23     585371        99.12
id_22     585371        99.12
id_27     585371        99.12
dist2     552913        93.63
D7        551623        93.41
id_18     545427        92.36
D13       528588        89.51
D14       528353        89.47
D12       525823        89.04
id_04     524216        88.77
id_03     524216        88.77
D6        517353        87.61
id_33     517251        87.59
id_09     515614        87.31


# Data Separation
მონაცემების 80/20 გაყოფა train და validation სეტებად. stratify პარამეტრით ვინარჩუნებთ fraud-ის 3.5% თანაფარდობას ორივე ნაწილში.

In [4]:
from sklearn.model_selection import train_test_split

y = train["isFraud"]
X = train.drop(columns=["isFraud", "TransactionID"])

X_train, X_val, y_train, y_val = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y,
)

print(f"X_train: {X_train.shape}")
print(f"X_val:   {X_val.shape}")
print(f"Fraud rate train: {y_train.mean():.4f}")
print(f"Fraud rate val:   {y_val.mean():.4f}")

del train
gc.collect()

X_train: (472432, 432)
X_val:   (118108, 432)
Fraud rate train: 0.0350
Fraud rate val:   0.0350


0

# Cleaning
ვშლით 90%-ზე მეტი null-ის მქონე სვეტებს. დანარჩენ null-ებისთვის რიცხვითებს ვავსებთ -999-ით, ხოლო კატეგორიილებს ტექსტით "missing"

In [5]:
from sklearn.preprocessing import LabelEncoder

missing_pct = X_train.isnull().sum() / len(X_train)
high_null_cols = missing_pct[missing_pct > 0.9].index.tolist()
print(f"Dropping {len(high_null_cols)} columns with >90% nulls")

X_train = X_train.drop(columns=high_null_cols)
X_val = X_val.drop(columns=high_null_cols)

num_cols = X_train.select_dtypes(include=[np.number]).columns.tolist()
cat_cols = X_train.select_dtypes(include=["object"]).columns.tolist()
print(f"Remaining: {len(num_cols)} numeric, {len(cat_cols)} categorical")

X_train[num_cols] = X_train[num_cols].fillna(-999)
X_val[num_cols] = X_val[num_cols].fillna(-999)

X_train[cat_cols] = X_train[cat_cols].fillna("missing")
X_val[cat_cols] = X_val[cat_cols].fillna("missing")

label_encoders = {}
for col in cat_cols:
    le = LabelEncoder()
    combined = pd.concat([X_train[col], X_val[col]], axis=0).astype(str)
    le.fit(combined)
    X_train[col] = le.transform(X_train[col].astype(str))
    X_val[col] = le.transform(X_val[col].astype(str))
    label_encoders[col] = le

print(f"Final shape: {X_train.shape}")
print(f"Remaining NaN: {X_train.isnull().sum().sum()}")
print(f"Remaining object cols: {(X_train.dtypes == 'object').sum()}")

Dropping 12 columns with >90% nulls
Remaining: 391 numeric, 29 categorical
Final shape: (472432, 420)
Remaining NaN: 0
Remaining object cols: 0


# Feature Engineering
უკვე არსებული ცვლადებიდან გამოგვყავს 7 ახალი ცვლადი, მაგალითად ტრანზაქციის საათი, მისი ათობითი ნაწილი, ბარათზე ტრანზაქციების საშუალო და ა.შ.

In [6]:
for df in [X_train, X_val]:
    df["Transaction_hour"] = (df["TransactionDT"] / 3600) % 24
    df["Transaction_dow"] = (df["TransactionDT"] / 86400) % 7
    df["TransactionAmt_log"] = np.log1p(df["TransactionAmt"])
    df["TransactionAmt_decimal"] = (df["TransactionAmt"] - df["TransactionAmt"].astype(int)).round(2)
    df["Card1_count"] = df["card1"].map(df["card1"].value_counts())
    df["Card1_TransactionAmt_mean"] = df["card1"].map(df.groupby("card1")["TransactionAmt"].mean())
    df["Amt_div_card1mean"] = df["TransactionAmt"] / (df["Card1_TransactionAmt_mean"] + 1)

new_features = [
    "Transaction_hour", "Transaction_dow", "TransactionAmt_log",
    "TransactionAmt_decimal", "Card1_count", "Card1_TransactionAmt_mean",
    "Amt_div_card1mean",
]
print(f"Added {len(new_features)} features")
print(f"X_train shape: {X_train.shape}")

Added 7 features
X_train shape: (472432, 427)


/tmp/ipykernel_57/2669828813.py:2: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["Transaction_hour"] = (df["TransactionDT"] / 3600) % 24
/tmp/ipykernel_57/2669828813.py:3: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["Transaction_dow"] = (df["TransactionDT"] / 86400) % 7
/tmp/ipykernel_57/2669828813.py:4: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To 

# Feature Selection
გამოვიყენოთ Lasso regularization, რომელიც კარგად მუშაობს წრფივი მოდელებისთვის.

In [7]:
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

scaler_for_selection = StandardScaler()
X_train_scaled_sel = scaler_for_selection.fit_transform(X_train)

lasso_selector = LogisticRegression(
    penalty="l1",
    solver="saga",
    C=0.01,
    max_iter=200,
    n_jobs=-1,
    random_state=42,
)
lasso_selector.fit(X_train_scaled_sel, y_train)

coefs = pd.Series(lasso_selector.coef_[0], index=X_train.columns)
selected_features = coefs[coefs != 0].index.tolist()

print(f"Selected {len(selected_features)} features (non-zero coefficients) from {X_train.shape[1]}")
print(f"\nTop 15 features by |coefficient|:")
print(coefs.abs().sort_values(ascending=False).head(15))

X_train_sel = X_train[selected_features]
X_val_sel = X_val[selected_features]

/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(


Selected 261 features (non-zero coefficients) from 427

Top 15 features by |coefficient|:
card6                     0.272080
M4                        0.271931
D5                        0.236211
TransactionAmt_log        0.226595
DeviceType                0.212502
D2                        0.187867
D1                        0.187006
C2                        0.174014
D13                       0.154383
TransactionAmt_decimal    0.152204
D4                        0.143576
C3                        0.134030
TransactionDT             0.130927
D8                        0.126864
C14                       0.126708
dtype: float64


# Training
ვატრენინგებთ logistic regression-ს სხვადასხვა კონფიგურაციით. ყველა მოდელისთვის ვიყენებთ StandardScaler-ს, რადგან linear მოდელებს scaling სჭირდებათ.

In [8]:
from sklearn.metrics import roc_auc_score
from sklearn.pipeline import Pipeline

def train_and_log_logreg(run_name, params, X_tr, X_va, y_tr, y_va):
    pipeline = Pipeline([
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(
            **params,
            random_state=42,
            n_jobs=-1,
        )),
    ])

    pipeline.fit(X_tr, y_tr)

    train_pred = pipeline.predict_proba(X_tr)[:, 1]
    val_pred = pipeline.predict_proba(X_va)[:, 1]

    train_auc = roc_auc_score(y_tr, train_pred)
    val_auc = roc_auc_score(y_va, val_pred)
    gap = train_auc - val_auc

    with mlflow.start_run(run_name=run_name):
        mlflow.log_params(params)
        mlflow.log_param("n_features", X_tr.shape[1])
        mlflow.log_metric("train_auc", train_auc)
        mlflow.log_metric("val_auc", val_auc)
        mlflow.log_metric("overfit_gap", gap)
        mlflow.sklearn.log_model(pipeline, name="model")

    print(f"{run_name:40s}  train_auc={train_auc:.4f}  val_auc={val_auc:.4f}  gap={gap:+.4f}")
    return pipeline, val_auc, val_pred

## Baseline
სტანდარტული Logistic Regression L2 regularization-ით.

In [9]:
params_baseline = {
    "penalty": "l2",
    "C": 1.0,
    "solver": "lbfgs",
    "max_iter": 500,
}

pipeline_baseline, auc_baseline, _ = train_and_log_logreg(
    "LogReg_baseline",
    params_baseline,
    X_train_sel, X_val_sel, y_train, y_val,
)

2026/05/02 13:12:31 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run LogReg_baseline at: https://dagshub.com/llikl23/IEEE-CIS-Fraud-Detection.mlflow/#/experiments/1/runs/2c5a2320b06b4d3b8e6040b03464736b
🧪 View experiment at: https://dagshub.com/llikl23/IEEE-CIS-Fraud-Detection.mlflow/#/experiments/1
LogReg_baseline                           train_auc=0.8311  val_auc=0.8306  gap=+0.0005


## Strong Regularization
ძლიერი L2 regularization-ით (C=0.01) შევამოწმებთ გვექნება თუ არა underfit.

In [10]:
params_strong_reg = {
    "penalty": "l2",
    "C": 0.01,
    "solver": "lbfgs",
    "max_iter": 500,
}

pipeline_strong, auc_strong, _ = train_and_log_logreg(
    "LogReg_strong_regularization",
    params_strong_reg,
    X_train_sel, X_val_sel, y_train, y_val,
)

2026/05/02 13:13:27 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run LogReg_strong_regularization at: https://dagshub.com/llikl23/IEEE-CIS-Fraud-Detection.mlflow/#/experiments/1/runs/b412b875029848d99dfe5c1c9454f57d
🧪 View experiment at: https://dagshub.com/llikl23/IEEE-CIS-Fraud-Detection.mlflow/#/experiments/1
LogReg_strong_regularization              train_auc=0.8235  val_auc=0.8264  gap=-0.0029


## Tuned + Class Weight Balancing
class_weight="balanced" ეხმარება მოდელს fraud-ის იშვიათ შემთხვევებზე ფოკუსირებაში.

In [11]:
params_tuned = {
    "penalty": "l2",
    "C": 0.5,
    "solver": "lbfgs",
    "max_iter": 1000,
    "class_weight": "balanced",
}

pipeline_tuned, auc_tuned, val_preds_tuned = train_and_log_logreg(
    "LogReg_tuned",
    params_tuned,
    X_train_sel, X_val_sel, y_train, y_val,
)

/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
2026/05/02 13:17:36 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run LogReg_tuned at: https://dagshub.com/llikl23/IEEE-CIS-Fraud-Detection.mlflow/#/experiments/1/runs/9d1f62e8f8534cef82961d26d995e7e6
🧪 View experiment at: https://dagshub.com/llikl23/IEEE-CIS-Fraud-Detection.mlflow/#/experiments/1
LogReg_tuned                              train_auc=0.8432  val_auc=0.8417  gap=+0.0015


## feature selection-ის გარეშე
ვამოწმებთ დაგვეხმარა თუ არა Lasso-ით feature selection-ი.

In [12]:
pipeline_all, auc_all, _ = train_and_log_logreg(
    "LogReg_tuned_all_features",
    params_tuned,
    X_train, X_val, y_train, y_val,
)

/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
2026/05/02 13:23:30 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run LogReg_tuned_all_features at: https://dagshub.com/llikl23/IEEE-CIS-Fraud-Detection.mlflow/#/experiments/1/runs/37d327b5a6ef4640a921499a8b61dcb8
🧪 View experiment at: https://dagshub.com/llikl23/IEEE-CIS-Fraud-Detection.mlflow/#/experiments/1
LogReg_tuned_all_features                 train_auc=0.8518  val_auc=0.8489  gap=+0.0028


# შედეგების შედარება

In [13]:
results = {
    "baseline (C=1.0)": auc_baseline,
    "strong_reg (C=0.01)": auc_strong,
    "tuned (C=0.5, balanced)": auc_tuned,
    "tuned + all features": auc_all,
}

print("=== LogReg Results ===")
for name, auc in sorted(results.items(), key=lambda x: x[1], reverse=True):
    print(f"  {name:40s}  val_auc={auc:.4f}")

best_name = max(results, key=results.get)
print(f"\nBest: {best_name} (AUC={results[best_name]:.4f})")

=== LogReg Results ===
  tuned + all features                      val_auc=0.8489
  tuned (C=0.5, balanced)                   val_auc=0.8417
  baseline (C=1.0)                          val_auc=0.8306
  strong_reg (C=0.01)                       val_auc=0.8264

Best: tuned + all features (AUC=0.8489)


## საუკეთესო LogReg მოდელის შენახვა
ვინახავთ საუკეთესო LogReg-ს. Model Registry-ში არ ვარეგისტრირებთ, რადგან XGBoost (0.97 AUC) გაცილებით უკეთეს შედეგებს იძლევა და საბოლოო registered მოდელი ის იქნება.

In [14]:
from sklearn.metrics import roc_auc_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

pipeline_final = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(
        penalty="l2",
        C=0.5,
        solver="lbfgs",
        max_iter=1000,
        class_weight="balanced",
        random_state=42,
        n_jobs=-1,
    )),
])

pipeline_final.fit(X_train, y_train)
val_pred = pipeline_final.predict_proba(X_val)[:, 1]
val_auc_final = roc_auc_score(y_val, val_pred)
print(f"Final LogReg val AUC = {val_auc_final:.4f}")

with mlflow.start_run(run_name="FINAL_LogReg_best") as run:
    mlflow.log_param("model", "LogisticRegression")
    mlflow.log_param("config", "tuned_all_features")
    mlflow.log_metric("val_auc", val_auc_final)
    mlflow.sklearn.log_model(pipeline_final, name="model")
    print("Logged FINAL with model artifact")

/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.
  n_iter_i = _check_optimize_result(


Final LogReg val AUC = 0.8489


2026/05/02 14:57:55 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format.


Logged FINAL with model artifact


# Pipeline
ვწვრთნით საბოლოო Logistic Regression Pipeline-ს, რომელიც preprocessing-ს და მოდელს აერთიანებს. ეს Pipeline პირდაპირ ეშვება დაუმუშავებელ test set-ზე.

In [15]:
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
import gc

class FraudPreprocessor(BaseEstimator, TransformerMixin):
    def __init__(self, null_threshold=0.9):
        self.null_threshold = null_threshold
    def fit(self, X, y=None):
        X = X.copy()
        X.columns = [c.replace("id-", "id_") for c in X.columns]
        if "TransactionID" in X.columns:
            X = X.drop(columns=["TransactionID"])
        missing_pct = X.isnull().sum() / len(X)
        self.high_null_cols_ = missing_pct[missing_pct > self.null_threshold].index.tolist()
        X = X.drop(columns=self.high_null_cols_)
        self.num_cols_ = X.select_dtypes(include=[np.number]).columns.tolist()
        self.cat_cols_ = X.select_dtypes(include=["object"]).columns.tolist()
        X[self.num_cols_] = X[self.num_cols_].fillna(-999)
        X[self.cat_cols_] = X[self.cat_cols_].fillna("missing")
        self.label_encoders_ = {}
        for col in self.cat_cols_:
            le = LabelEncoder()
            le.fit(X[col].astype(str))
            self.label_encoders_[col] = le
        self.card1_counts_ = X["card1"].value_counts().to_dict()
        self.card1_amt_mean_ = X.groupby("card1")["TransactionAmt"].mean().to_dict()
        for col in self.cat_cols_:
            X[col] = self.label_encoders_[col].transform(X[col].astype(str))
        X = self._add_features(X)
        self.feature_columns_ = X.columns.tolist()
        return self
    def transform(self, X):
        X = X.copy()
        X.columns = [c.replace("id-", "id_") for c in X.columns]
        if "TransactionID" in X.columns:
            X = X.drop(columns=["TransactionID"])
        cols_to_drop = [c for c in self.high_null_cols_ if c in X.columns]
        X = X.drop(columns=cols_to_drop)
        for col in self.num_cols_:
            if col in X.columns: X[col] = X[col].fillna(-999)
        for col in self.cat_cols_:
            if col in X.columns: X[col] = X[col].fillna("missing")
        for col in self.cat_cols_:
            if col in X.columns:
                le = self.label_encoders_[col]
                known = set(le.classes_); fallback = le.classes_[0]
                X[col] = X[col].astype(str).apply(lambda v: v if v in known else fallback)
                X[col] = le.transform(X[col])
        X = self._add_features(X)
        for col in self.feature_columns_:
            if col not in X.columns: X[col] = 0
        return X[self.feature_columns_]
    def _add_features(self, X):
        X["Transaction_hour"] = (X["TransactionDT"] / 3600) % 24
        X["Transaction_dow"] = (X["TransactionDT"] / 86400) % 7
        X["TransactionAmt_log"] = np.log1p(X["TransactionAmt"])
        X["TransactionAmt_decimal"] = (X["TransactionAmt"] - X["TransactionAmt"].astype(int)).round(2)
        X["Card1_count"] = X["card1"].map(self.card1_counts_).fillna(0)
        X["Card1_TransactionAmt_mean"] = X["card1"].map(self.card1_amt_mean_).fillna(X["TransactionAmt"].mean())
        X["Amt_div_card1mean"] = X["TransactionAmt"] / (X["Card1_TransactionAmt_mean"] + 1)
        return X

DATA_DIR = "/kaggle/input/competitions/ieee-fraud-detection"
train_t = pd.read_csv(f"{DATA_DIR}/train_transaction.csv")
train_i = pd.read_csv(f"{DATA_DIR}/train_identity.csv")
train_full = train_t.merge(train_i, on="TransactionID", how="left")
del train_t, train_i
gc.collect()

y_full = train_full["isFraud"]
X_full = train_full.drop(columns=["isFraud"])
del train_full
gc.collect()

X_tr_raw, X_va_raw, y_tr, y_va = train_test_split(
    X_full, y_full, test_size=0.2, random_state=42, stratify=y_full,
)
del X_full
gc.collect()

logreg_pipeline = Pipeline([
    ("preprocessor", FraudPreprocessor(null_threshold=0.9)),
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(
        penalty="l2", C=0.5, solver="lbfgs", max_iter=1000,
        class_weight="balanced", random_state=42, n_jobs=-1,
    )),
])

print("Fitting LogReg Pipeline on raw data...")
logreg_pipeline.fit(X_tr_raw, y_tr)
val_pred = logreg_pipeline.predict_proba(X_va_raw)[:, 1]
val_auc = roc_auc_score(y_va, val_pred)
print(f"LogReg Pipeline Val AUC: {val_auc:.4f}")

with mlflow.start_run(run_name="LogReg_Pipeline_FINAL") as run:
    mlflow.log_param("model", "LogReg_Pipeline")
    mlflow.log_metric("val_auc", val_auc)
    mlflow.sklearn.log_model(logreg_pipeline, name="model")
    print("LogReg Pipeline logged")

Fitting LogReg Pipeline on raw data...


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.
  n_iter_i = _check_optimize_result(


LogReg Pipeline Val AUC: 0.8491


2026/05/02 16:08:27 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format.


LogReg Pipeline logged
🏃 View run LogReg_Pipeline_FINAL at: https://dagshub.com/llikl23/IEEE-CIS-Fraud-Detection.mlflow/#/experiments/1/runs/11aca961543e4460babb86d2b39bfcdf
🧪 View experiment at: https://dagshub.com/llikl23/IEEE-CIS-Fraud-Detection.mlflow/#/experiments/1
